# Python MD From Scratch

This notebook builds a minimal molecular dynamics engine in Python. The goal is not speed; the goal is to see the moving parts of an MD method clearly.

You will implement:

1. Particle positions and velocities.
2. Lennard-Jones pair forces.
3. Periodic boundary conditions.
4. Velocity Verlet integration.
5. Basic energy and temperature analysis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

## 1. Initialize a Small 2D System

We will place particles on a square grid and assign small random velocities. Reduced units are used throughout: mass, `sigma`, `epsilon`, and Boltzmann constant are all set to 1.

In [ ]:
rng = np.random.default_rng(4)

n_side = 5
n_atoms = n_side ** 2
box_length = 8.0
spacing = box_length / n_side

x = np.linspace(0.5 * spacing, box_length - 0.5 * spacing, n_side)
xx, yy = np.meshgrid(x, x)
positions = np.column_stack([xx.ravel(), yy.ravel()])

velocities = rng.normal(scale=0.2, size=(n_atoms, 2))
velocities -= velocities.mean(axis=0)

plt.scatter(positions[:, 0], positions[:, 1])
plt.xlim(0, box_length)
plt.ylim(0, box_length)
plt.gca().set_aspect("equal")
plt.title("Initial Particle Positions")
plt.show()

## 2. Minimum-Image Periodic Boundaries

Periodic boundary conditions make a small simulation cell behave like part of a larger repeating material. The minimum-image convention uses the nearest periodic image of each particle pair.

In [ ]:
def minimum_image(displacement, box_length):
    return displacement - box_length * np.round(displacement / box_length)


def wrap_positions(positions, box_length):
    return positions % box_length

## 3. Lennard-Jones Forces

The Lennard-Jones potential is a common model for simple nonbonded interactions. The code below computes pair forces and potential energy for all unique pairs.

In [ ]:
def compute_lj_forces(positions, box_length, cutoff=2.5):
    forces = np.zeros_like(positions)
    potential = 0.0
    cutoff2 = cutoff ** 2

    for i in range(len(positions) - 1):
        for j in range(i + 1, len(positions)):
            rij = minimum_image(positions[j] - positions[i], box_length)
            r2 = np.dot(rij, rij)

            if r2 < cutoff2:
                inv_r2 = 1.0 / r2
                inv_r6 = inv_r2 ** 3
                inv_r12 = inv_r6 ** 2
                potential += 4.0 * (inv_r12 - inv_r6)
                force_vector = 24.0 * (2.0 * inv_r12 - inv_r6) * inv_r2 * rij
                forces[i] -= force_vector
                forces[j] += force_vector

    return forces, potential

## 4. Run the MD Simulation

Velocity Verlet updates positions using current velocities and accelerations, then updates velocities using the average of old and new accelerations.

In [ ]:
dt = 0.004
n_steps = 1500
mass = 1.0

trajectory = np.zeros((n_steps, n_atoms, 2))
energies = np.zeros((n_steps, 3))

forces, potential = compute_lj_forces(positions, box_length)

for step in range(n_steps):
    trajectory[step] = positions

    kinetic = 0.5 * mass * np.sum(velocities ** 2)
    energies[step] = [kinetic + potential, kinetic, potential]

    accelerations = forces / mass
    positions = positions + velocities * dt + 0.5 * accelerations * dt ** 2
    positions = wrap_positions(positions, box_length)

    new_forces, potential = compute_lj_forces(positions, box_length)
    new_accelerations = new_forces / mass
    velocities = velocities + 0.5 * (accelerations + new_accelerations) * dt
    forces = new_forces

In [ ]:
time = np.arange(n_steps) * dt

plt.plot(time, energies[:, 0], label="total")
plt.plot(time, energies[:, 1], label="kinetic")
plt.plot(time, energies[:, 2], label="potential")
plt.xlabel("Time")
plt.ylabel("Energy")
plt.title("Energy Conservation Check")
plt.legend()
plt.show()

In [ ]:
plt.scatter(trajectory[0, :, 0], trajectory[0, :, 1], label="start", alpha=0.7)
plt.scatter(trajectory[-1, :, 0], trajectory[-1, :, 1], label="final", alpha=0.7)
plt.xlim(0, box_length)
plt.ylim(0, box_length)
plt.gca().set_aspect("equal")
plt.title("Initial and Final Configurations")
plt.legend()
plt.show()

## Exercises

1. Change `dt` and observe the total energy drift.
2. Increase `n_side` and notice how runtime changes.
3. Add a simple thermostat by rescaling velocities.
4. Save the trajectory to an XYZ-style file.
5. Replace the pair loop with a neighbor-list strategy conceptually.